# 05 — Batch Prediction

Inferencia masiva de clientes usando el pipeline entrenado.

## Objetivos

- Cargar nuevos datos para inferencia.
- Validar estructura y calidad de entrada.
- Generar probabilidades y clases predichas.
- Aplicar umbrales configurables.
- Priorizar clientes para campañas de retención.
- Exportar resultados auditables.
- Medir tiempos y volúmenes de procesamiento.


## 1. Configuración del entorno

Ejecute el notebook desde la raíz del repositorio.

En Google Colab:

```python
!git clone <URL_DEL_REPOSITORIO>
%cd customer-intelligence-ml-platform
```


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if (candidate / "src").exists():
            PROJECT_ROOT = candidate
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


## 2. Importaciones


In [ ]:
import json
import time
import joblib
import numpy as np
import pandas as pd

from src.data import load_customer_data
from src.models import load_model, predict_customers
from src.utils import get_project_path, get_logger

logger = get_logger("notebook.batch_prediction")


## 3. Rutas de entrada y salida


In [ ]:
INPUT_PATH = get_project_path(
    "data",
    "raw",
    "customer_churn.csv",
)

MODEL_PATH = get_project_path(
    "artifacts",
    "models",
    "selected_churn_pipeline.joblib",
)

PREDICTIONS_PATH = get_project_path(
    "reports",
    "predictions",
    "customer_predictions.csv",
    create_parent=True,
)

SUMMARY_PATH = get_project_path(
    "reports",
    "metrics",
    "batch_prediction_summary.json",
    create_parent=True,
)

print(INPUT_PATH)
print(MODEL_PATH)
print(PREDICTIONS_PATH)
print(SUMMARY_PATH)


## 4. Carga de datos para inferencia


In [ ]:
input_df = load_customer_data(
    INPUT_PATH,
    validate=True,
)

print(f"Rows: {len(input_df):,}")
print(f"Columns: {input_df.shape[1]}")
input_df.head()


Aunque el dataset de ejemplo contiene `churn`, en producción la variable objetivo no estará disponible al momento de predecir.


In [ ]:
inference_df = input_df.drop(
    columns=["churn"],
    errors="ignore",
).copy()

print(inference_df.shape)
inference_df.head()


## 5. Validación de columnas


In [ ]:
required_columns = {
    "customer_id",
    "gender",
    "age",
    "region",
    "customer_segment",
    "contract_type",
    "tenure_months",
    "monthly_fee",
    "total_spent",
    "internet_service",
    "tv_service",
    "streaming_service",
    "support_calls",
    "complaints",
    "payment_method",
    "last_payment_delay",
    "digital_usage_score",
    "marketing_score",
    "preferred_contact_hour",
}

missing_columns = sorted(
    required_columns - set(inference_df.columns)
)

extra_columns = sorted(
    set(inference_df.columns) - required_columns
)

{
    "missing_columns": missing_columns,
    "extra_columns": extra_columns,
}


In [ ]:
if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("Input schema validation passed.")


## 6. Calidad de datos de entrada


In [ ]:
quality_summary = pd.DataFrame({
    "dtype": inference_df.dtypes.astype(str),
    "missing_count": inference_df.isna().sum(),
    "missing_pct": (
        inference_df.isna().sum()
        / len(inference_df)
        * 100
    ),
    "unique_values": inference_df.nunique(
        dropna=True
    ),
})

quality_summary.sort_values(
    "missing_pct",
    ascending=False,
).head(15)


In [ ]:
duplicate_ids = int(
    inference_df[
        "customer_id"
    ].duplicated().sum()
)

if duplicate_ids > 0:
    raise ValueError(
        f"Found {duplicate_ids} duplicated customer IDs."
    )

print("No duplicated customer IDs.")


## 7. Carga del pipeline


In [ ]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        "No se encontró el modelo entrenado. "
        "Ejecute primero 02_model_development.ipynb."
    )

pipeline = load_model(MODEL_PATH)

print(type(pipeline))
print(pipeline.named_steps.keys())


## 8. Predicción batch con umbral base


In [ ]:
DEFAULT_THRESHOLD = 0.50

start_time = time.perf_counter()

prediction_df = predict_customers(
    pipeline,
    inference_df,
    threshold=DEFAULT_THRESHOLD,
)

elapsed_seconds = (
    time.perf_counter()
    - start_time
)

prediction_df.head()


In [ ]:
print(f"Predicted rows: {len(prediction_df):,}")
print(f"Elapsed seconds: {elapsed_seconds:.4f}")
print(
    "Rows per second: "
    f"{len(prediction_df) / elapsed_seconds:,.2f}"
)


## 9. Integración con atributos de negocio


In [ ]:
business_columns = [
    "customer_id",
    "region",
    "customer_segment",
    "contract_type",
    "monthly_fee",
    "tenure_months",
    "support_calls",
    "complaints",
    "last_payment_delay",
]

enriched_predictions = (
    inference_df[business_columns]
    .merge(
        prediction_df,
        on="customer_id",
        how="left",
        validate="one_to_one",
    )
)

enriched_predictions.head()


## 10. Segmentación por nivel de riesgo


In [ ]:
def assign_risk_band(
    probability: float,
) -> str:
    if probability >= 0.75:
        return "Muy alto"
    if probability >= 0.55:
        return "Alto"
    if probability >= 0.35:
        return "Medio"
    return "Bajo"

enriched_predictions["risk_band"] = (
    enriched_predictions[
        "churn_probability"
    ].apply(assign_risk_band)
)

enriched_predictions[
    "risk_band"
].value_counts()


## 11. Priorización comercial


In [ ]:
enriched_predictions[
    "retention_priority"
] = np.select(
    [
        enriched_predictions[
            "churn_probability"
        ] >= 0.75,
        enriched_predictions[
            "churn_probability"
        ] >= 0.55,
        enriched_predictions[
            "churn_probability"
        ] >= 0.35,
    ],
    [
        1,
        2,
        3,
    ],
    default=4,
)

priority_table = (
    enriched_predictions
    .sort_values(
        [
            "retention_priority",
            "churn_probability",
            "monthly_fee",
        ],
        ascending=[
            True,
            False,
            False,
        ],
    )
)

priority_table.head(20)


## 12. Cupo operativo de campaña


In [ ]:
CAMPAIGN_CAPACITY = 150

campaign_list = (
    priority_table
    .head(CAMPAIGN_CAPACITY)
    .copy()
)

print(
    f"Campaign capacity: {CAMPAIGN_CAPACITY}"
)
print(
    "Average churn probability: "
    f"{campaign_list['churn_probability'].mean():.2%}"
)
print(
    "Estimated monthly revenue at risk: "
    f"S/ {campaign_list['monthly_fee'].sum():,.2f}"
)


## 13. Comparación de umbrales


In [ ]:
thresholds = [
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
]

threshold_summary = []

probabilities = prediction_df[
    "churn_probability"
].to_numpy()

for threshold in thresholds:
    selected = (
        probabilities >= threshold
    )

    threshold_summary.append({
        "threshold": threshold,
        "selected_customers": int(
            selected.sum()
        ),
        "selected_pct": float(
            selected.mean()
        ),
        "estimated_monthly_fee_at_risk": float(
            inference_df.loc[
                selected,
                "monthly_fee",
            ].sum()
        ),
    })

threshold_summary_df = pd.DataFrame(
    threshold_summary
)

threshold_summary_df


## 14. Análisis por región


In [ ]:
regional_summary = (
    enriched_predictions
    .groupby(
        "region",
        dropna=False,
    )
    .agg(
        customers=(
            "customer_id",
            "count",
        ),
        average_probability=(
            "churn_probability",
            "mean",
        ),
        predicted_churn_rate=(
            "churn_prediction",
            "mean",
        ),
        monthly_fee_at_risk=(
            "monthly_fee",
            lambda series: series[
                enriched_predictions.loc[
                    series.index,
                    "churn_prediction",
                ] == 1
            ].sum(),
        ),
    )
    .sort_values(
        "average_probability",
        ascending=False,
    )
)

regional_summary


## 15. Análisis por tipo de contrato


In [ ]:
contract_summary = (
    enriched_predictions
    .groupby(
        "contract_type",
        dropna=False,
    )
    .agg(
        customers=(
            "customer_id",
            "count",
        ),
        average_probability=(
            "churn_probability",
            "mean",
        ),
        predicted_churn_rate=(
            "churn_prediction",
            "mean",
        ),
    )
    .sort_values(
        "predicted_churn_rate",
        ascending=False,
    )
)

contract_summary


## 16. Validación de probabilidades


In [ ]:
probability_checks = {
    "minimum": float(
        prediction_df[
            "churn_probability"
        ].min()
    ),
    "maximum": float(
        prediction_df[
            "churn_probability"
        ].max()
    ),
    "mean": float(
        prediction_df[
            "churn_probability"
        ].mean()
    ),
    "outside_range": int(
        (
            ~prediction_df[
                "churn_probability"
            ].between(0, 1)
        ).sum()
    ),
}

probability_checks


## 17. Validación de consistencia


In [ ]:
consistency_checks = {
    "same_row_count": (
        len(inference_df)
        == len(prediction_df)
    ),
    "unique_output_ids": (
        prediction_df[
            "customer_id"
        ].is_unique
    ),
    "valid_prediction_values": set(
        prediction_df[
            "churn_prediction"
        ].unique()
    ).issubset({0, 1}),
    "probabilities_in_range": (
        prediction_df[
            "churn_probability"
        ].between(0, 1).all()
    ),
}

consistency_checks


In [ ]:
if not all(
    consistency_checks.values()
):
    raise RuntimeError(
        "Batch prediction validation failed."
    )

print("Batch prediction validation passed.")


## 18. Exportación de predicciones


In [ ]:
PREDICTIONS_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

priority_table.to_csv(
    PREDICTIONS_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(PREDICTIONS_PATH)


## 19. Exportación de lista de campaña


In [ ]:
CAMPAIGN_PATH = get_project_path(
    "reports",
    "predictions",
    "retention_campaign.csv",
    create_parent=True,
)

campaign_list.to_csv(
    CAMPAIGN_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(CAMPAIGN_PATH)


## 20. Resumen operativo


In [ ]:
batch_summary = {
    "input_file": str(
        INPUT_PATH
    ),
    "model_file": str(
        MODEL_PATH
    ),
    "rows_received": int(
        len(inference_df)
    ),
    "rows_predicted": int(
        len(prediction_df)
    ),
    "threshold": float(
        DEFAULT_THRESHOLD
    ),
    "predicted_churn_count": int(
        prediction_df[
            "churn_prediction"
        ].sum()
    ),
    "predicted_churn_rate": float(
        prediction_df[
            "churn_prediction"
        ].mean()
    ),
    "average_churn_probability": float(
        prediction_df[
            "churn_probability"
        ].mean()
    ),
    "processing_seconds": float(
        elapsed_seconds
    ),
    "rows_per_second": float(
        len(prediction_df)
        / elapsed_seconds
    ),
    "campaign_capacity": int(
        CAMPAIGN_CAPACITY
    ),
    "consistency_checks": {
        key: bool(value)
        for key, value in consistency_checks.items()
    },
}

batch_summary


## 21. Exportación del resumen


In [ ]:
SUMMARY_PATH.write_text(
    json.dumps(
        batch_summary,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(SUMMARY_PATH)


## 22. Lectura de resultados exportados


In [ ]:
exported_predictions = pd.read_csv(
    PREDICTIONS_PATH
)

exported_campaign = pd.read_csv(
    CAMPAIGN_PATH
)

print(
    exported_predictions.shape
)
print(
    exported_campaign.shape
)

exported_campaign.head()


## 23. Simulación de archivo nuevo


In [ ]:
SAMPLE_INPUT_PATH = get_project_path(
    "data",
    "external",
    "customer_batch_sample.csv",
    create_parent=True,
)

inference_df.head(50).to_csv(
    SAMPLE_INPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(SAMPLE_INPUT_PATH)


## 24. Predicción sobre archivo externo


In [ ]:
external_df = pd.read_csv(
    SAMPLE_INPUT_PATH
)

external_predictions = predict_customers(
    pipeline,
    external_df,
    threshold=0.50,
)

external_predictions.head()


## 25. Manejo de un archivo defectuoso


In [ ]:
broken_external_df = (
    external_df
    .drop(
        columns=[
            "monthly_fee",
        ]
    )
)

try:
    predict_customers(
        pipeline,
        broken_external_df,
    )
except Exception as error:
    print(type(error).__name__)
    print(error)


## 26. Consideraciones para producción

Antes de automatizar batch prediction, se debería incorporar:

- identificación de versión del modelo;
- fecha y hora de inferencia;
- validación formal del esquema;
- logging estructurado;
- almacenamiento del archivo original;
- control de duplicados;
- monitoreo del volumen;
- alertas por datos atípicos;
- trazabilidad entre entrada y salida.


## 27. Preguntas para estudiantes


1. ¿Por qué conviene conservar `customer_id` en la salida?
2. ¿Qué diferencia existe entre probabilidad, clase y banda de riesgo?
3. ¿Cómo elegiría el tamaño de la campaña?
4. ¿Qué ocurriría si se cambiara el umbral a 0.30?
5. ¿Qué validaciones deberían realizarse antes de llamar al modelo?
6. ¿Qué columnas debería contener un archivo batch real?
7. ¿Cómo aseguraría trazabilidad entre entrada y predicción?


## 28. Checklist de cierre


- [ ] Se validó el esquema de entrada.
- [ ] Se verificaron IDs duplicados.
- [ ] Se cargó correctamente el pipeline.
- [ ] Se generaron probabilidades.
- [ ] Se asignaron clases predichas.
- [ ] Se crearon bandas de riesgo.
- [ ] Se priorizó la campaña.
- [ ] Se compararon umbrales.
- [ ] Se analizaron regiones y contratos.
- [ ] Se validó consistencia de salida.
- [ ] Se exportaron predicciones.
- [ ] Se exportó la campaña.
- [ ] Se guardó el resumen operativo.
- [ ] Se probó un archivo externo.
- [ ] Se simuló un error de esquema.


## Resultado esperado

El notebook transforma el pipeline entrenado en un proceso batch reproducible.

La salida principal es una tabla priorizada con:

- identificador del cliente;
- probabilidad de churn;
- clase predicha;
- banda de riesgo;
- prioridad comercial;
- variables relevantes para la campaña.
